# 🧪 Démo Colab — est-ce que j'ai bien un GPU NVIDIA ?

Petit notebook pour **vérifier** que tout marche. `Run All`, et tu sais.
Il tourne **partout** avec le même code : ton CPU, ton Radeon 880M, ou un T4 chez Google.

## Se connecter au T4

**`Select Kernel`** (en haut à droite) → `Select Another Kernel...` → `Colab` →
connexion Google → **`New Colab Server`** → **`GPU`** → **`T4`**

> ⚠️ **Le GPU se choisit à la création du serveur, nulle part ailleurs.**
> Si tu prends `Auto Connect` ou que tu oublies l'étape `GPU`, tu récupères un
> **serveur CPU** — plus lent que ton propre PC, pour zéro bénéfice. Un serveur créé en
> CPU le reste : il faut en refaire un.
>
> Le menu `Runtime → Change runtime type` des tutos n'existe **que sur le Colab web**.

## Tes 3 noyaux

| Choix | Matériel |
|---|---|
| `Python 3` | ton CPU |
| `Python (GPU ROCm)` | ton Radeon 880M |
| `Colab` | NVIDIA T4 distant |

## 🔒 Ça ne touche pas à ton Drive

Les données sont lues par **une URL GitHub** : Colab télécharge 3,8 Ko et rien d'autre.
Aucune autorisation, rien de monté. Détails en section 2.

## 1. Où je tourne, et sur quoi ?

In [ ]:
import torch

try:
    import google.colab
    DANS_COLAB = True
except ImportError:
    DANS_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("J'execute sur :", "Google Colab (a distance)" if DANS_COLAB else "ton PC")
print(f"torch         : {torch.__version__}")
print(f"device        : {device}")

if device.type == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU           : {torch.cuda.get_device_name(0)}  ({vram:.1f} Go de VRAM)")
    print("\n>>> GPU OK.")
elif DANS_COLAB:
    print("\n>>> Colab t'a donne un serveur CPU, pas un GPU.")
    print("    Select Kernel -> Select Another Kernel... -> Colab")
    print("    -> New Colab Server -> GPU -> T4")
else:
    print("\n>>> Noyau CPU. Pour ton Radeon : Select Kernel -> Python (GPU ROCm)")

## 2. Les données

**Une seule source de vérité : ton dépôt GitHub.** La même URL sur ton PC et sur Colab,
donc le même fichier et le même résultat, toujours. Aucun clic.

```
donnees/Advertising.csv  →  tu le pousses  →  le notebook le relit par son URL
```

### ⚠️ La seule règle

> **Tu modifies un CSV dans `donnees/` → tu pousses avant de relancer.**

Sinon le notebook lit l'ancienne version **sans rien dire**. Les commandes sont dans
`donnees/commandes_terminal_pour_push_donnees.txt`.

### Ce qui ne marche PAS, et pourquoi

| | Pourquoi |
|---|---|
| `files.upload()` | Fenêtre **JavaScript** qui exige l'interface **web** de Colab. Dans VS Code : widget mort, cellule bloquée à l'infini. Aucun contournement. |
| `drive.mount()` | Monterait ton « Mon Drive » **en entier** — photos, vidéos, documents. Impossible de le limiter à un dossier. |
| Un chemin local | La machine Colab est **vide**. Seul le **code** part, pas ton dossier. |

> 💡 **Le réflexe** : avant de lancer un notebook que tu n'as pas écrit, cherche
> `drive.mount` dedans. S'il y est, il aura accès à **tout ton Drive** pendant qu'il tourne.
>
> Ce que tu as autorisé :
> [myaccount.google.com/permissions](https://myaccount.google.com/permissions)

In [ ]:
import pandas as pd

# Une seule source : ton depot GitHub public.
# Colab telecharge 3,8 Ko par HTTP et rien d'autre. Ton Drive n'est jamais touche.
GITHUB = ("https://raw.githubusercontent.com/RobinsanKiritheepan/"
          "Cours_Python/main/Cours_IA/Deep_learning/Google_Colab/donnees")


def source(nom):
    """L'URL du fichier sur ton depot GitHub."""
    return f"{GITHUB}/{nom}"


print("Lecture depuis GitHub :")
print(source("Advertising.csv"))

df = pd.read_csv(source("Advertising.csv"))
print(f"\n{len(df)} lignes, {len(df.columns)} colonnes")
df.head()

## 3. Le GPU calcule-t-il vraiment ?

Détecter un GPU et **s'en servir** sont deux choses différentes. On entraîne un petit
réseau et on regarde si la perte descend.

In [ ]:
import time

from torch import nn

torch.manual_seed(42)

# .to(device) : c'est CA qui envoie les donnees sur le GPU. Rien ne part tout seul.
X = torch.tensor(df[["TV", "radio", "newspaper"]].values, dtype=torch.float32)
y = torch.tensor(df[["sales"]].values, dtype=torch.float32)
X = ((X - X.mean(0)) / X.std(0)).to(device)   # normalisation, sinon ca n'apprend pas
y = y.to(device)

modele = nn.Sequential(
    nn.Linear(3, 32), nn.ReLU(),
    nn.Linear(32, 1),
).to(device)                                  # <- le modele part sur le GPU

perte_fn = nn.MSELoss()
opt = torch.optim.Adam(modele.parameters(), lr=0.05)

t0 = time.perf_counter()
for epoch in range(500):
    opt.zero_grad()
    perte = perte_fn(modele(X), y)
    perte.backward()
    opt.step()
    if epoch == 0:
        perte_debut = perte.item()
if device.type == "cuda":
    torch.cuda.synchronize()
duree = time.perf_counter() - t0

print(f"perte : {perte_debut:.2f}  ->  {perte.item():.2f}")
print(f"500 epochs en {duree:.2f} s sur {device}")
print(f"le modele est sur : {next(modele.parameters()).device}")
print(f"nb de parametres  : {sum(p.numel() for p in modele.parameters())}")
print("\n>>> " + ("OK, ca apprend." if perte.item() < perte_debut
                 else "ECHEC : la perte ne descend pas."))

## 4. Le verdict

In [ ]:
ou = "Google Colab" if DANS_COLAB else "ton PC"
materiel = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"

print("=" * 52)
print(f"  Endroit  : {ou}")
print(f"  Materiel : {materiel}")
print(f"  Vitesse  : {duree:.2f} s pour 500 epochs")
print(f"  Donnees  : GitHub (ton Drive jamais monte)")
print("=" * 52)
print("\nNote la vitesse, change de noyau, relance tout, compare.")

## 5. À quoi t'attendre

**Ce réseau fait 161 paramètres. Le GPU sera PLUS LENT que ton CPU.**
Ce n'est pas un bug : le temps d'envoyer les données vers la carte dépasse le temps de
calcul.

**Mesuré sur ta machine le 16/07/2026** (500 epochs, ce notebook) :

| Noyau | Temps |
|---|---|
| `Python 3` (CPU) | **0,27 s** |
| `Python (GPU ROCm)` (Radeon 880M) | **1,09 s** ← **4× plus lent** |

La perte est identique des deux côtés (211,60 → 0,20) : ton code est reproductible, seule
la vitesse change.

Un GPU, c'est un poids lourd : inutile pour livrer une lettre. Il gagne sur les **CNN sur
images**, les **transformers**, et les **batchs ≥ 32**.

| Ce que tu fais | Utilise |
|---|---|
| cette démo, CSV, sklearn, petits réseaux | **ton CPU** (`Python 3`) |
| CNN sur images, transformers | **Colab (T4)** |
| tester ton Radeon | `Python (GPU ROCm)` |

**Cette démo ne sert pas à aller vite** — elle sert à vérifier que **la plomberie marche**
avant que tu en aies vraiment besoin. Le jour du premier CNN, tout sera déjà en place.

---

### ⚠️ Pièges connus

**1.** Le noyau `Python (GPU ROCm)` **se fige au démarrage de temps en temps** (0 % de CPU,
rien ne s'affiche). Stack ROCm sur Windows, en preview. `Restart Kernel` et ça repart.

**2.** L'auth de l'extension Colab a des bugs ouverts (`Exchange timeout exceeded`).
Réessaie, ça finit par passer.

**3.** Si tu modifies ce notebook pendant qu'il est ouvert ailleurs, VS Code peut garder
l'ancienne version en mémoire (un **`●`** apparaît sur l'onglet). Ferme sans enregistrer,
puis rouvre.